# Slovenia Solvency reports Table S.02.01.02; Phase 3 Cross-Validation

This script is a continuation of `Phase_2_Processing_Slovenia_SII_demo`.

The purpose of this script is to perform the third and final automated stage of the S.02.01.02 transcribing. Cross-Validation takes the generated table and performs a series of internal consistency checks. First on an aggregate level and then for each company in the dataset. The functions return a None if the check is passed or the fields that are compared if the check fails. This allows the user to quickly identify the mistakes in the transcribing. 

The way we operate this notebook is that we first look at the summary of all companies. If the entire array is true (all tests pass for all companies), then the process moves into the final phase. If not, the user looks further down through the notebook at individual companies. Where a test fails, the relevant fields are displayed. The user than compares the pdf with the values displayed. If any corrections are necessary, the user does them above the test in code. In this way, when the script is run again, the tests passes.


## Description of the process

The process of extraction is performed in 5 phases:

### Phase 0: Find the reports and identify the relevant tables. 
 1) Identify the new SFCR report and save it into the folder Input.
 2) Identify the pages where the tables of interest are.
 3) Compile the map of the company run in the master_list.csv.

### Phase 1: Run the Extraction script. 
The script performs the following steps (with slight modifications depending on the table format):
 1) Save the page with the table into a separate folder Single_pdf.
 2) Use either a Python package or specialized LLM to create a digital equivalent of the table.
 3) Fix the systemic errors that prevent the table from being saved as DataFrame.
 4) Save the DataFrame into the Output folder.

### Phase 2: Run the Processing script. 
The script applies fixes to the DataFrame to make the numbers closer to the reported numbers. It joins all the tables into a single dataset and saves it into the Dirty_Combined folder. 

### Phase 3: Run the Cross-Validation script (this script). 
Applies a series of tests that check for the internal consistency between the numbers. Flags potential errors. After the individual fixes are applied, it saves the table into the Cleaner_Combined folder.

### Phase 4: Final modifications to the table and a manual inspection. 

### Tolerance

In [357]:
eps = 1.5

#### Necessary packages

In [358]:
import pandas as pd
import numpy as np

### Functions

Functions that start with "check_" implement a single internal consistency check. Comparing two or more numbers if they match.

In [359]:
def check_02_01_02_1(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0100 = R0110 + R0120
    
    """

    lhs = data.loc["R0100",col]
    rhs = data.loc["R0110",col] + data.loc["R0120",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        return False, pd.DataFrame(data = [data.loc["R0110",col], data.loc["R0120",col], data.loc["R0100",col]], index=["R0110", "R0120","R0100"], columns = ["TEST_1"]), diff 

In [360]:
def check_02_01_02_2(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0130 = R0140 + R0150 + R0160 + R0170
    
    """

    lhs = data.loc["R0130",col]
    rhs = data.loc["R0140",col] + data.loc["R0150",col]+ data.loc["R0160",col]+ data.loc["R0170",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0140",col], data.loc["R0150",col], data.loc["R0160",col], data.loc["R0170",col], data.loc["R0130",col]]
        dagnostics_index = ["R0140", "R0150", "R0160", "R0170", "R0130"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_2"]), diff 

In [361]:
def check_02_01_02_3(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0070 = R0080 + R0090 + R0100 + R0130 + R0180 + R0190 + R0200 + R0210
    
    """

    lhs = data.loc["R0070",col]
    rhs = data.loc["R0080",col] + data.loc["R0090",col]+ data.loc["R0100",col]+ data.loc["R0130",col] + data.loc["R0180",col]+ data.loc["R0190",col]+ data.loc["R0200",col]+ data.loc["R0210",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0080",col], data.loc["R0090",col], data.loc["R0100",col], data.loc["R0130",col],
                      data.loc["R0180",col], data.loc["R0190",col], data.loc["R0200",col], data.loc["R0210",col], data.loc["R0070",col]]
            
        dagnostics_index = ["R0080", "R0090", "R0100", "R0130", "R0180", "R0190", "R0200","R0210", "R0070"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_3"]), diff 

In [362]:
def check_02_01_02_4(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0230 = R0240 + R0250 + R0260

    """

    lhs = data.loc["R0230",col]
    rhs = data.loc["R0240",col] + data.loc["R0250",col]+ data.loc["R0260",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0240",col], data.loc["R0250",col], data.loc["R0260",col], data.loc["R0230",col]]
        dagnostics_index = ["R0240", "R0250", "R0260", "R0230"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_4"]), diff 

In [363]:
def check_02_01_02_5(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0270 = R0280 + R0310 + R0340

    """

    lhs = data.loc["R0270",col]
    rhs = data.loc["R0280",col] + data.loc["R0310",col]+ data.loc["R0340",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0280",col], data.loc["R0310",col], data.loc["R0340",col], data.loc["R0270",col]]
        dagnostics_index = ["R0280", "R0310", "R0340", "R0270"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_5"]), diff 

In [364]:
def check_02_01_02_6(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0310 = R0320 + R0330
    
    """

    lhs = data.loc["R0310",col]
    rhs = data.loc["R0320",col] + data.loc["R0330",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0320",col], data.loc["R0330",col], data.loc["R0310",col]]
        dagnostics_index = ["R0320", "R0330", "R0310"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_6"]), diff 

In [365]:
def check_02_01_02_7(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0280 = R0290 + R0300
    
    """

    lhs = data.loc["R0280",col]
    rhs = data.loc["R0290",col] + data.loc["R0300",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0290",col], data.loc["R0300",col], data.loc["R0280",col]]
        dagnostics_index = ["R0290", "R0300", "R0280"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_7"]), diff 

In [366]:
def check_02_01_02_8(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0500 = R0030 + R0040 + R0050 + R0060 + R0070 + R0220 + R0230 + 
            R0270 + R0350 + R0360 + R0370 + R0380 + R0390 + R0400 + R0410 + R0420

    """

    lhs = data.loc["R0500",col]
    rhs = data.loc["R0030",col] + data.loc["R0040",col] + data.loc["R0050",col] + data.loc["R0060",col] + data.loc["R0070",col] + data.loc["R0220",col] + data.loc["R0230",col] + data.loc["R0270",col] + data.loc["R0350",col] + data.loc["R0360",col]+ data.loc["R0370",col]+ data.loc["R0380",col]+ data.loc["R0390",col]+ data.loc["R0400",col]+ data.loc["R0410",col]+ data.loc["R0420",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0030",col], data.loc["R0040",col], data.loc["R0050",col], data.loc["R0060",col], data.loc["R0070",col], data.loc["R0220",col], data.loc["R0230",col], data.loc["R0270",col], data.loc["R0350",col], data.loc["R0360",col], data.loc["R0370",col], data.loc["R0380",col], data.loc["R0390",col], data.loc["R0400",col], data.loc["R0410",col], data.loc["R0420",col], data.loc["R0500",col]]
        dagnostics_index = ["R0030", "R0040", "R0050", "R0060", "R0070", "R0220", "R0230", "R0270", "R0350", "R0360", "R0370", "R0380", "R0390", "R0400", "R0410", "R0420", "R0500"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_8"]), diff 

In [367]:
def check_02_01_02_9(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0510 = R0520 + R0560
    
    """

    lhs = data.loc["R0510",col]
    rhs = data.loc["R0520",col] + data.loc["R0560",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0520",col], data.loc["R0560",col], data.loc["R0510",col]]
        dagnostics_index = ["R0520", "R0560", "R0510"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_9"]), diff 

In [368]:
def check_02_01_02_10(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0520 = R0530 + R0540 + R0550
    
    """

    lhs = data.loc["R0520",col]
    rhs = data.loc["R0530",col] + data.loc["R0540",col]+ data.loc["R0550",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0530",col], data.loc["R0540",col], data.loc["R0550",col], data.loc["R0520",col]]
        dagnostics_index = ["R0530", "R0540", "R0550", "R0520"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_10"]), diff 

In [369]:
def check_02_01_02_11(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0560 = R0570 + R0580 + R0590
    
    """

    lhs = data.loc["R0560",col]
    rhs = data.loc["R0570",col] + data.loc["R0580",col]+ data.loc["R0590",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0570",col], data.loc["R0580",col], data.loc["R0590",col], data.loc["R0560",col]]
        dagnostics_index = ["R0570", "R0580", "R0590", "R0560"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_11"]), diff 

In [370]:
def check_02_01_02_12(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0600 = R0610 + R0650
    
    """

    lhs = data.loc["R0600",col]
    rhs = data.loc["R0610",col] + data.loc["R0650",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0610",col], data.loc["R0650",col], data.loc["R0600",col]]
        dagnostics_index = ["R0610", "R0650", "R0600"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_12"]), diff 

In [371]:
def check_02_01_02_13(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0610 = R0620 + R0630 + R0640
    
    """

    lhs = data.loc["R0610",col]
    rhs = data.loc["R0620",col] + data.loc["R0630",col]+ data.loc["R0640",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0620",col], data.loc["R0630",col], data.loc["R0640",col], data.loc["R0610",col]]
        dagnostics_index = ["R0620", "R0630", "R0640", "R0610"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_13"]), diff 

In [372]:
def check_02_01_02_14(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0650 = R0660 + R0670 + R0680
    
    """

    lhs = data.loc["R0650",col]
    rhs = data.loc["R0660",col] + data.loc["R0670",col]+ data.loc["R0680",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0660",col], data.loc["R0670",col], data.loc["R0680",col], data.loc["R0650",col]]
        dagnostics_index = ["R0660", "R0670", "R0680", "R0650"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_14"]), diff 

In [373]:
def check_02_01_02_15(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0690 = R0700 + R0710 + R0720
    
    """

    lhs = data.loc["R0690",col]
    rhs = data.loc["R0700",col] + data.loc["R0710",col]+ data.loc["R0720",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0700",col], data.loc["R0710",col], data.loc["R0720",col], data.loc["R0690",col]]
        dagnostics_index = ["R0700", "R0710", "R0720", "R0690"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_15"]), diff 

In [374]:
def check_02_01_02_16(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R0850 = R0860 + R0870
    
    """

    lhs = data.loc["R0850",col]
    rhs = data.loc["R0860",col] + data.loc["R0870",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0860",col], data.loc["R0870",col], data.loc["R0850",col]]
        dagnostics_index = ["R0860", "R0870", "R0850"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_16"]), diff 

In [375]:
def check_02_01_02_17(data:pd.DataFrame, eps:float, col:str) -> list[bool, pd.DataFrame, float]:
    """
    R1000 = R0500 - R0900
    
    """

    lhs = data.loc["R1000",col]
    rhs = data.loc["R0500",col] - data.loc["R0900",col]
    diff = abs(lhs-rhs)
    if diff< eps:
        return True, None, diff
    else:
        diagnostics = [data.loc["R0500",col], data.loc["R0900",col], data.loc["R1000",col]]
        dagnostics_index = ["R0500", "R0900", "R1000"]
        return False, pd.DataFrame(data = diagnostics, index=dagnostics_index, columns = ["TEST_17"]), diff 

In [376]:
def run_check_diagnostics(table: pd.DataFrame, function, col_name: str, eps: float) -> pd.DataFrame:
    """
    Runs a single check provided as a function for every column of a dataframe.
    
    """
    results = pd.DataFrame(data=[], columns=["COMPANY_NAME",col_name])
    for col in table.columns:
        res, diag, observed_tol = function(table, eps=eps, col=col)
        result_tmp = pd.DataFrame(data=[[col, res]], columns = ["COMPANY_NAME",col_name])
        results = pd.concat([results, result_tmp])
        results[results == True] =""
        
    return results

## Dirty table

Upolads to memory the dirty data table from the previous phase.

In [377]:
dirty_table =  pd.read_csv("Dirty_Combined/Table_slo_S02.csv", header=0, index_col=0)

In [378]:
dirty_table = dirty_table.loc[dirty_table.index.notna()]

In [379]:
dirty_table.drop(index=["CODE"], inplace=True)

In [380]:
dirty_table.drop(columns=["MASTER"], inplace=True)

In [381]:
dirty_table = dirty_table.astype(float).fillna(0)

## High-level overview of quality

Performs the checks for all companies at the same time. Assessing the overall quality of the table before looking at individual companies.

In [382]:
results_1 = run_check_diagnostics(dirty_table, check_02_01_02_1, "TEST_1", eps)
results_1 = results_1.set_index("COMPANY_NAME")

In [383]:
results_2 = run_check_diagnostics(dirty_table, check_02_01_02_2, "TEST_2", eps)
results_2 = results_2.set_index("COMPANY_NAME")

In [384]:
results_3 = run_check_diagnostics(dirty_table, check_02_01_02_3, "TEST_3", eps)
results_3 = results_3.set_index("COMPANY_NAME")

In [385]:
results_4 = run_check_diagnostics(dirty_table, check_02_01_02_4, "TEST_4", eps)
results_4 = results_4.set_index("COMPANY_NAME")

In [386]:
results_5 = run_check_diagnostics(dirty_table, check_02_01_02_5, "TEST_5", eps)
results_5 = results_5.set_index("COMPANY_NAME")

In [387]:
results_6 = run_check_diagnostics(dirty_table, check_02_01_02_6, "TEST_6", eps)
results_6 = results_6.set_index("COMPANY_NAME")

In [388]:
results_7 = run_check_diagnostics(dirty_table, check_02_01_02_7, "TEST_7", eps)
results_7 = results_7.set_index("COMPANY_NAME")

In [389]:
results_8 = run_check_diagnostics(dirty_table, check_02_01_02_8, "TEST_8", eps)
results_8 = results_8.set_index("COMPANY_NAME")

In [390]:
results_9 = run_check_diagnostics(dirty_table, check_02_01_02_9, "TEST_9", eps)
results_9 = results_9.set_index("COMPANY_NAME")

In [391]:
results_10 = run_check_diagnostics(dirty_table, check_02_01_02_10, "TEST_10", eps)
results_10 = results_10.set_index("COMPANY_NAME")

In [392]:
results_11 = run_check_diagnostics(dirty_table, check_02_01_02_11, "TEST_11", eps)
results_11 = results_11.set_index("COMPANY_NAME")

In [393]:
results_12 = run_check_diagnostics(dirty_table, check_02_01_02_12, "TEST_12", eps)
results_12 = results_12.set_index("COMPANY_NAME")

In [394]:
results_13 = run_check_diagnostics(dirty_table, check_02_01_02_13, "TEST_13", eps)
results_13 = results_13.set_index("COMPANY_NAME")

In [395]:
results_14 = run_check_diagnostics(dirty_table, check_02_01_02_14, "TEST_14", eps)
results_14 = results_14.set_index("COMPANY_NAME")

In [396]:
results_15 = run_check_diagnostics(dirty_table, check_02_01_02_15, "TEST_15", eps)
results_15 = results_15.set_index("COMPANY_NAME")

In [397]:
results_16 = run_check_diagnostics(dirty_table, check_02_01_02_16, "TEST_16", eps)
results_16 = results_16.set_index("COMPANY_NAME")

In [398]:
results_17 = run_check_diagnostics(dirty_table, check_02_01_02_17, "TEST_17", eps)
results_17 = results_17.set_index("COMPANY_NAME")

In [399]:
overall_summary = pd.concat([results_1,results_2,results_3,results_4,results_5,results_6,results_7,results_8,results_9,results_10,results_11,
                             results_12,results_13,results_14,results_15,results_16,results_17], axis=1)

In [400]:
display(overall_summary)

,TEST_1,TEST_2,TEST_3,TEST_4,TEST_5,TEST_6,TEST_7,TEST_8,TEST_9,TEST_10,TEST_11,TEST_12,TEST_13,TEST_14,TEST_15,TEST_16,TEST_17
COMPANY_NAME,,,,,,,,,,,,,,,,,
MODRA,,,,,,,,,,,,False,,False,,,
TRIGLAV,,,,,,,,,,,,False,False,False,,,False
PRVA,,,,,,,,False,,,,False,,False,False,,False
GENERALI,,,,,,,,,,,,,,,,,
VZAJEMNA,,,,,,,,,,,,,,,,,
GRAWE,,,,,,,,,False,,,,,,False,,
SAVA,,,,,,,,,,,,,,,,,


## Manual overwrites

If a check is failing, modify the numbers at the begining of the section until the tests are passed.

Note that sometimes small errors are due to rounding.

## Modra zavarovalnica d.d.

In [402]:
unique_code = "MODRA"

In [403]:
dirty_table.loc["R0650",unique_code] = 382508.46

In [404]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [405]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [406]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [407]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [408]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [409]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [410]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [411]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [412]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [413]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [414]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [415]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [416]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [417]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [418]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [419]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [420]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## TRIGLAV

In [421]:
unique_code = "TRIGLAV"

In [422]:
dirty_table.loc["R0650",unique_code] = 699614
dirty_table.loc["R0600",unique_code] = 699761
dirty_table.loc["R0640",unique_code] = 0
dirty_table.loc["R0660",unique_code] = 0
dirty_table.loc["R0670",unique_code] = 678631
dirty_table.loc["R0680",unique_code] = 20983
dirty_table.loc["R0900",unique_code] = 2471275
dirty_table.loc["R1000",unique_code] = 1044800

In [423]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [424]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [425]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [426]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [427]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [428]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [429]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [430]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [431]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [432]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [433]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [434]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [435]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [436]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [437]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [438]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [439]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## Prva osebna zavarovalnica, d.d.

In [440]:
unique_code = "PRVA"

In [441]:
dirty_table.loc["R0500",unique_code] = 51250
dirty_table.loc["R0610",unique_code] = 59
dirty_table.loc["R0650",unique_code] = 3040
dirty_table.loc["R0630",unique_code] = -408
dirty_table.loc["R0640",unique_code] = 466
dirty_table.loc["R0710",unique_code] = 14810

In [442]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [443]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [444]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [445]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [446]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [447]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [448]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [449]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [450]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [451]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [452]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [453]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [454]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [455]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [456]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [457]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [458]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## GENERALI

In [459]:
unique_code = "GENERALI"

In [460]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [461]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [462]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [463]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [464]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [465]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [466]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [467]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [468]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [469]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [470]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [471]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [472]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [473]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [474]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [475]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [476]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## Vzajemna d.d.

In [477]:
unique_code = "VZAJEMNA"

In [478]:
dirty_table.loc["R0510",unique_code] = 13908

In [479]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [480]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [481]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [482]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [483]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [484]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [485]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [486]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [487]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [488]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [489]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [490]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [491]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [492]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [493]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [494]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [495]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## Grawe zavarovalnica d.d.

In [496]:
unique_code = "GRAWE"

In [497]:
dirty_table.loc["R0510",unique_code] = 28104
dirty_table.loc["R0720",unique_code] = 1950

In [498]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [499]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [500]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [501]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [502]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [503]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [504]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [505]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [506]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [507]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [508]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [509]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [510]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [511]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [512]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [513]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [514]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## Zavarovalnica Sava, d.d.

In [515]:
unique_code = "SAVA"

In [516]:
result,diagnostics, diff = check_02_01_02_1(dirty_table, eps, unique_code)
diagnostics

In [517]:
result,diagnostics, diff = check_02_01_02_2(dirty_table, eps, unique_code)
diagnostics

In [518]:
result,diagnostics, diff = check_02_01_02_3(dirty_table, eps, unique_code)
diagnostics

In [519]:
result,diagnostics, diff = check_02_01_02_4(dirty_table, eps, unique_code)
diagnostics

In [520]:
result,diagnostics, diff = check_02_01_02_5(dirty_table, eps, unique_code)
diagnostics

In [521]:
result,diagnostics, diff = check_02_01_02_6(dirty_table, eps, unique_code)
diagnostics

In [522]:
result,diagnostics, diff = check_02_01_02_7(dirty_table, eps, unique_code)
diagnostics

In [523]:
result,diagnostics, diff = check_02_01_02_8(dirty_table, eps, unique_code)
diagnostics

In [524]:
result,diagnostics, diff = check_02_01_02_9(dirty_table, eps, unique_code)
diagnostics

In [525]:
result,diagnostics, diff = check_02_01_02_10(dirty_table, eps, unique_code)
diagnostics

In [526]:
result,diagnostics, diff = check_02_01_02_11(dirty_table, eps, unique_code)
diagnostics

In [527]:
result,diagnostics, diff = check_02_01_02_12(dirty_table, eps, unique_code)
diagnostics

In [528]:
result,diagnostics, diff = check_02_01_02_13(dirty_table, eps, unique_code)
diagnostics

In [529]:
result,diagnostics, diff = check_02_01_02_14(dirty_table, eps, unique_code)
diagnostics

In [530]:
result,diagnostics, diff = check_02_01_02_15(dirty_table, eps, unique_code)
diagnostics

In [531]:
result,diagnostics, diff = check_02_01_02_16(dirty_table, eps, unique_code)
diagnostics

In [532]:
result,diagnostics, diff = check_02_01_02_17(dirty_table, eps, unique_code)
diagnostics

## Master table after cross checks

Saves the final data table in the folder `Clearer_Combined`.


In [533]:
dirty_table.to_csv("Cleaner_Combined/Table_slo_S02.csv")